<a href="https://colab.research.google.com/github/trialdyk/marchine-learning/blob/main/JS05/JS05_TUGAS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Lab 2 - Multinomial Naive Bayes: CountVectorizer vs TF-IDF

Praktikum Pembelajaran Mesin - JS05 Klasifikasi 1

**Tugas:**
1. Buatlah model Multinomial Naive Bayes dengan `spam.csv` menggunakan `CountVectorizer` dengan `stop_words` diaktifkan, lalu evaluasi performanya.
2. Buatlah model Multinomial Naive Bayes dengan `spam.csv` menggunakan `TF-IDF` dengan `stop_words` diaktifkan. Evaluasi hasilnya, bandingkan dengan tugas 1, lalu tentukan fitur mana yang bekerja paling baik untuk klasifikasi spam.


## Persiapan - Load dan Preprocessing Data

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('spam.csv', encoding='latin-1') # spesifiksi encoding diperlukan karena data tidak menggunakan UTF-8

# Drop 3 kolom terakhir dengan fungsi iloc
df = df.drop(df.iloc[:,2:], axis=1)

# Ganti nama kolom (v1, v2 -> Labels, SMS)
df.columns = ['Labels', 'SMS']

df.head()

,Labels,SMS
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
# Cek Jumlah Data Per Kelas
print(df['Labels'].value_counts())
print('\n')

# Cek Kelengkapan Data
print(df.info())

Labels
ham     4825
spam     747
Name: count, dtype: int64


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Labels  5572 non-null   object
 1   SMS     5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


In [3]:
# Encode label
new_labels = {
    'spam': 1,
    'ham': 0
}
df['Labels'] = df['Labels'].map(new_labels)

X = df['SMS'].values
y = df['Labels'].values

In [4]:
from sklearn.model_selection import train_test_split

# Split data training dan testing (sama untuk kedua model agar perbandingannya adil)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

## Tugas 1 - Multinomial NB dengan CountVectorizer (stop_words aktif)

`stop_words='english'` membuang kata umum bahasa Inggris (misalnya *the*, *is*, *and*) karena dataset SMS ini berbahasa Inggris.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Fitting dan transform data training, hanya transform untuk data testing
bow = CountVectorizer(stop_words='english')
X_train_bow = bow.fit_transform(X_train_raw)
X_test_bow = bow.transform(X_test_raw)

print(f'Dimensi data: {X_train_bow.shape}')

# Model
mnb_bow = MultinomialNB()
mnb_bow.fit(X_train_bow, y_train)

# Evaluasi
y_pred_train_bow = mnb_bow.predict(X_train_bow)
y_pred_test_bow = mnb_bow.predict(X_test_bow)
acc_train_bow = accuracy_score(y_train, y_pred_train_bow)
acc_test_bow = accuracy_score(y_test, y_pred_test_bow)

print(f'Hasil akurasi data train: {acc_train_bow}')
print(f'Hasil akurasi data test: {acc_test_bow}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_test_bow))
print('Laporan Klasifikasi:\n', classification_report(y_test, y_pred_test_bow, target_names=['ham', 'spam']))

Dimensi data: (4457, 7466)
Hasil akurasi data train: 0.9946152120260264
Hasil akurasi data test: 0.9829596412556054
Confusion Matrix:
 [[951   3]
 [ 16 145]]
Laporan Klasifikasi:
               precision    recall  f1-score   support

         ham       0.98      1.00      0.99       954
        spam       0.98      0.90      0.94       161

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



## Tugas 2 - Multinomial NB dengan TF-IDF (stop_words aktif)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train_raw)
X_test_tfidf = tfidf.transform(X_test_raw)

print(f'Dimensi data: {X_train_tfidf.shape}')

# Model
mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)

# Evaluasi
y_pred_train_tfidf = mnb_tfidf.predict(X_train_tfidf)
y_pred_test_tfidf = mnb_tfidf.predict(X_test_tfidf)
acc_train_tfidf = accuracy_score(y_train, y_pred_train_tfidf)
acc_test_tfidf = accuracy_score(y_test, y_pred_test_tfidf)

print(f'Hasil akurasi data train (TF-IDF): {acc_train_tfidf}')
print(f'Hasil akurasi data test (TF-IDF): {acc_test_tfidf}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_test_tfidf))
print('Laporan Klasifikasi:\n', classification_report(y_test, y_pred_test_tfidf, target_names=['ham', 'spam']))

Dimensi data: (4457, 7466)
Hasil akurasi data train (TF-IDF): 0.9842943684092439
Hasil akurasi data test (TF-IDF): 0.9605381165919282
Confusion Matrix:
 [[954   0]
 [ 44 117]]
Laporan Klasifikasi:
               precision    recall  f1-score   support

         ham       0.96      1.00      0.98       954
        spam       1.00      0.73      0.84       161

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115



## Perbandingan CountVectorizer vs TF-IDF

Pada spam, kesalahan yang paling merugikan biasanya *ham* yang terbuang sebagai spam (false positive), maka selain akurasi, perhatikan juga precision dan recall kelas spam.

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score

def ringkas(nama, acc_train, y_true, y_pred, acc_test):
    return {
        'Fitur': nama,
        'Akurasi train': acc_train,
        'Akurasi test': acc_test,
        'Precision spam': precision_score(y_true, y_pred),
        'Recall spam': recall_score(y_true, y_pred),
        'F1 spam': f1_score(y_true, y_pred)
    }

perbandingan = pd.DataFrame([
    ringkas('CountVectorizer', acc_train_bow, y_test, y_pred_test_bow, acc_test_bow),
    ringkas('TF-IDF', acc_train_tfidf, y_test, y_pred_test_tfidf, acc_test_tfidf),
]).set_index('Fitur')
perbandingan

,Akurasi train,Akurasi test,Precision spam,Recall spam,F1 spam
Fitur,,,,,
CountVectorizer,0.994615,0.982960,0.97973,0.900621,0.938511
TF-IDF,0.984294,0.960538,1.00000,0.726708,0.841727


Hasil di atas berasal dari satu kali pembagian data. Sebagai pengecekan tambahan, bandingkan juga dengan cross-validation 5-fold pada seluruh data (vectorizer berada di dalam pipeline sehingga tidak ada kebocoran data).

In [8]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

pipe_bow = make_pipeline(CountVectorizer(stop_words='english'), MultinomialNB())
pipe_tfidf = make_pipeline(TfidfVectorizer(stop_words='english'), MultinomialNB())

for nama, pipe in [('CountVectorizer', pipe_bow), ('TF-IDF', pipe_tfidf)]:
    skor_acc = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
    skor_f1 = cross_val_score(pipe, X, y, cv=5, scoring='f1')
    print(f'{nama:16s} akurasi CV: {skor_acc.mean():.4f} (+/- {skor_acc.std():.4f}) | F1 spam CV: {skor_f1.mean():.4f}')

CountVectorizer  akurasi CV: 0.9856 (+/- 0.0019) | F1 spam CV: 0.9453
TF-IDF           akurasi CV: 0.9682 (+/- 0.0045) | F1 spam CV: 0.8654


### Jawaban Tugas 2

**Fitur yang bekerja paling baik:** _(isi: CountVectorizer atau TF-IDF, sesuai tabel perbandingan dan hasil cross-validation di atas)_

**Alasan / analisis:** _(jelaskan, misalnya: selisih akurasi, precision/recall kelas spam, dan mengapa pembobotan TF-IDF membantu atau tidak untuk Multinomial NB pada data SMS ini)_